# 05 Peer Comparison Engine
## Nifty Financial Platform

This notebook allows for a detailed comparison of a specific company against its sector peers. We calculate relative rankings and percentiles for key financial metrics.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load environment variables
load_dotenv(dotenv_path='../.env')
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    print("ERROR: DATABASE_URL not found in .env file")
else:
    engine = create_engine(DATABASE_URL)
    print("Database engine created successfully.")

## 1. Load Data
We need company, sector, and financial data.

In [ ]:
def fetch_peer_data(engine):
    query = """
    SELECT 
        dc.symbol, dc.company_name, ds.sector_name, 
        pl.net_profit_margin_pct, 
        bs.debt_to_equity,
        an.stock_pe, an.roce_pct, an.market_cap
    FROM fact_profit_loss pl
    JOIN dim_company dc ON pl.company_id = dc.company_id
    JOIN dim_sector ds ON dc.sector_id = ds.sector_id
    JOIN fact_balance_sheet bs ON pl.company_id = bs.company_id AND pl.year_id = bs.year_id
    JOIN fact_analysis an ON pl.company_id = an.company_id AND pl.year_id = an.year_id
    WHERE pl.year_id = (SELECT MAX(year_id) FROM fact_profit_loss)
    """
    return pd.read_sql(query, engine)

df = fetch_peer_data(engine)
print(f"Loaded {len(df)} companies for comparison.")

## 2. Peer Comparison Function
A reusable function to compare a company with its sector.

In [ ]:
def compare_company(symbol, df):
    if symbol not in df['symbol'].values:
        print(f"Symbol {symbol} not found.")
        return
    
    target = df[df['symbol'] == symbol].iloc[0]
    sector = target['sector_name']
    peers = df[df['sector_name'] == sector]
    
    metrics = ['net_profit_margin_pct', 'debt_to_equity', 'stock_pe', 'roce_pct', 'market_cap']
    
    print(f"Comparing {target['company_name']} ({symbol}) in Sector: {sector}")
    print(f"Number of peers: {len(peers)}")
    
    comparison = []
    for m in metrics:
        val = target[m]
        avg = peers[m].mean()
        rank = peers[m].rank(ascending=(False if m != 'debt_to_equity' and m != 'stock_pe' else True)).loc[target.name]
        pct = (peers[m] < val).mean() * 100
        comparison.append({'Metric': m, 'Value': val, 'Sector Avg': avg, 'Rank': int(rank), 'Percentile': pct})
        
    return pd.DataFrame(comparison)

# Example Comparison
test_symbol = df['symbol'].iloc[0]
comp_df = compare_company(test_symbol, df)
comp_df

## 3. Visualize Comparison (Radar Chart)
Normalized radar chart to show performance across multiple dimensions.

In [ ]:
def plot_radar(symbol, df):
    target = df[df['symbol'] == symbol].iloc[0]
    sector = target['sector_name']
    peers = df[df['sector_name'] == sector]
    
    metrics = ['net_profit_margin_pct', 'roce_pct', 'market_cap'] # Use select metrics for radar
    
    fig = go.Figure()
    
    # Normalize values for plotting (0-1 range based on sector)
    target_vals = []
    sector_avg_vals = []
    
    for m in metrics:
        m_min = peers[m].min()
        m_max = peers[m].max()
        target_vals.append((target[m] - m_min) / (m_max - m_min) if m_max != m_min else 0.5)
        sector_avg_vals.append((peers[m].mean() - m_min) / (m_max - m_min) if m_max != m_min else 0.5)

    fig.add_trace(go.Scatterpolar(r=target_vals, theta=metrics, fill='toself', name=symbol))
    fig.add_trace(go.Scatterpolar(r=sector_avg_vals, theta=metrics, fill='toself', name='Sector Avg'))

    fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 1])), showlegend=True, title=f"{symbol} vs {sector} Peers")
    fig.show()

plot_radar(test_symbol, df)

## 4. Peer Ranking Tables

In [ ]:
target = df[df['symbol'] == test_symbol].iloc[0]
peers = df[df['sector_name'] == target['sector_name']].sort_values('market_cap', ascending=False)
print(f"Top Peers in {target['sector_name']} by Market Cap:")
display(peers[['symbol', 'company_name', 'market_cap', 'roce_pct', 'net_profit_margin_pct']].head(10))